# 13 — JNLPBA (biomedical) Arm 1: prep + zero-shot + few-shot fine-tuning

Adds the biomedical domain (JNLPBA; types `protein, DNA, RNA, cell_line, cell_type`) as a third
target, **without touching any existing notebook or result file**. This one notebook does, for
JNLPBA only, what Notebooks 02/03/04/05 do for WNUT-17 and SciERC:

1. deterministic few-shot splits (50/100/200 × seeds 13/42/101),
2. label map + tokenization (first-subword alignment, `max_length` 256),
3. zero-shot cross-domain evaluation of the CoNLL baseline (entity-boundary F1),
4. few-shot fine-tuning (Arm 1) — same recipe as Notebook 05.

All JNLPBA outputs go under **`results/jnlpba/`** and `tokenized/jnlpba/…`, so the WNUT/SciERC
results are untouched. Notebook 14 (LLM) and 15 (aggregation) build on these.

In [1]:
!pip -q install "transformers==4.44.2" "datasets==2.19.2" "seqeval==1.2.2" "accelerate>=0.26.0" pandas

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 2.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 3.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 69.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 542.1/542.1 kB 35.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 172.0/172.0 kB 15.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 25.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 65.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2024.3.1 which is incompatible.
gradio 6.20.0 requires huggingface-hub<2.0,>=1.2.0, but you have huggingface-hub 0.36.2 which is incompatible.


## Step 1 — Config

In [2]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

import json, random
from pathlib import Path
import numpy as np, pandas as pd

PROCESSED      = Path('/content/drive/MyDrive/AAI590/data/processed')
TOKENIZED_DIR  = PROCESSED / 'tokenized'
LABELS_DIR     = PROCESSED / 'label_maps'
MODELS_DIR     = PROCESSED / 'models'
FEWSHOT_SPLITS = PROCESSED / 'fewshot_splits'
BASELINE_DIR   = MODELS_DIR / 'baseline_conll2003'

DS = 'jnlpba'
BUDGETS = [50, 100, 200]
SEEDS   = [13, 42, 101]
MAX_LENGTH = 256

JN_RESULTS = PROCESSED / 'results' / 'jnlpba'
JN_RESULTS.mkdir(parents=True, exist_ok=True)

def load_jsonl(p):
    return [json.loads(l) for l in open(p) if l.strip()]
def save_jsonl(p, rows):
    Path(p).parent.mkdir(parents=True, exist_ok=True)
    with open(p, 'w') as f:
        for r in rows: f.write(json.dumps(r, ensure_ascii=True) + '\n')

assert (PROCESSED / DS / f'{DS}_train.jsonl').exists(), 'Run 00b first (jnlpba not found).'
assert BASELINE_DIR.exists(), 'Baseline (Notebook 04) not found.'
print('jnlpba train/val/test:',
      *[len(load_jsonl(PROCESSED / DS / f'{DS}_{s}.jsonl')) for s in ['train','validation','test']])

Mounted at /content/drive
jnlpba train/val/test: 16807 1739 3856


## Step 2 — Deterministic few-shot splits

Identical stratified sampler to `02_make_fewshot_splits` (80/20 entity/non-entity, same seeds),
so JNLPBA's few-shot subsets are constructed the same way as the other domains.

In [3]:
def sent_has_entity(s): return any(t != 'O' for t in s['tags'])

def stratified_sample_train(samples, budget, seed):
    rng = random.Random(seed)
    pos = [s for s in samples if sent_has_entity(s)]
    neg = [s for s in samples if not sent_has_entity(s)]
    rng.shuffle(pos); rng.shuffle(neg)
    n_pos = min(len(pos), max(1, int(round(0.8 * budget))))
    n_neg = min(len(neg), budget - n_pos)
    chosen = pos[:n_pos] + neg[:n_neg]
    if len(chosen) < budget:
        used = set(id(x) for x in chosen)
        left = [s for s in samples if id(s) not in used]; rng.shuffle(left)
        chosen.extend(left[:budget - len(chosen)])
    rng.shuffle(chosen)
    return chosen[:budget]

train = load_jsonl(PROCESSED / DS / f'{DS}_train.jsonl')
for b in BUDGETS:
    for s in SEEDS:
        fp = FEWSHOT_SPLITS / DS / f'{DS}_train_{b}_seed_{s}.jsonl'
        if fp.exists(): continue
        save_jsonl(fp, stratified_sample_train(train, b, s))
print('few-shot splits ready ->', FEWSHOT_SPLITS / DS)

few-shot splits ready -> /content/drive/MyDrive/AAI590/data/processed/fewshot_splits/jnlpba


## Step 3 — Label map + tokenization (same as Notebook 03)

In [4]:
from transformers import AutoTokenizer
from datasets import Dataset

def build_label_list():
    tags = set()
    for split in ('train', 'validation', 'test'):
        for r in load_jsonl(PROCESSED / DS / f'{DS}_{split}.jsonl'):
            tags.update(r['tags'])
    return ['O'] + sorted(t for t in tags if t != 'O')

label_list = build_label_list()
label2id = {l: i for i, l in enumerate(label_list)}
id2label = {i: l for l, i in label2id.items()}
LABELS_DIR.mkdir(parents=True, exist_ok=True)
json.dump({'label2id': label2id, 'id2label': id2label},
          open(LABELS_DIR / f'{DS}_label_map.json', 'w'), indent=2)
print(DS, '->', len(label_list), 'labels:', label_list)

tokenizer = AutoTokenizer.from_pretrained(str(BASELINE_DIR))

def tok_align(rows):
    def enc(batch):
        e = tokenizer(batch['tokens'], is_split_into_words=True, truncation=True, max_length=MAX_LENGTH)
        labs = []
        for i, tags in enumerate(batch['tags']):
            prev = None; lab = []
            for w in e.word_ids(batch_index=i):
                lab.append(-100 if (w is None or w == prev) else label2id[tags[w]]); prev = w
            labs.append(lab)
        e['labels'] = labs; return e
    return Dataset.from_list([{'tokens': r['tokens'], 'tags': r['tags']} for r in rows]) \
        .map(enc, batched=True, remove_columns=['tokens', 'tags'])

for split in ('validation', 'test'):
    tok_align(load_jsonl(PROCESSED / DS / f'{DS}_{split}.jsonl')).save_to_disk(str(TOKENIZED_DIR / DS / split))
for b in BUDGETS:
    for s in SEEDS:
        rows = load_jsonl(FEWSHOT_SPLITS / DS / f'{DS}_train_{b}_seed_{s}.jsonl')
        tok_align(rows).save_to_disk(str(TOKENIZED_DIR / 'fewshot' / DS / f'budget{b}_seed{s}'))
print('tokenized ->', TOKENIZED_DIR / DS)

jnlpba -> 11 labels: ['O', 'B-DNA', 'B-RNA', 'B-cell_line', 'B-cell_type', 'B-protein', 'I-DNA', 'I-RNA', 'I-cell_line', 'I-cell_type', 'I-protein']


Map:   0%|          | 0/1739 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/1739 [00:00<?, ? examples/s]

Map:   0%|          | 0/3856 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/3856 [00:00<?, ? examples/s]

...

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/200 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/200 [00:00<?, ? examples/s]

tokenized -> /content/drive/MyDrive/AAI590/data/processed/tokenized/jnlpba


## Step 4 — Metrics (seqeval, identical to Notebooks 05/09)

In [5]:
from seqeval.metrics import precision_score, recall_score, f1_score, classification_report

def collapse(t): return 'O' if t == 'O' else ('B-ENT' if t.startswith('B-') else 'I-ENT')

def decode(preds, labels, idmap):
    preds = np.argmax(preds, axis=2); tt, pp = [], []
    for pr, lr in zip(preds, labels):
        st, sp = [], []
        for p, l in zip(pr, lr):
            if l == -100: continue
            st.append(idmap[int(l)]); sp.append(idmap[int(p)])
        tt.append(st); pp.append(sp)
    return tt, pp

def per_type_from(tt, pp):
    rep = classification_report(tt, pp, output_dict=True, zero_division=0)
    return {k: {'precision': float(v['precision']), 'recall': float(v['recall']),
                'f1': float(v['f1-score']), 'support': int(v['support'])}
            for k, v in rep.items() if k not in ('micro avg', 'macro avg', 'weighted avg')}

## Step 5 — Zero-shot cross-domain evaluation of the CoNLL baseline

The CoNLL model has no biomedical labels, so we score it entity-boundary-only (all typed tags
collapsed to a single entity class), exactly as in Notebook 04. Saved to
`results/jnlpba/baseline_jnlpba.json`.

In [6]:
import torch
from transformers import AutoModelForTokenClassification, DataCollatorForTokenClassification
from datasets import load_from_disk

collator = DataCollatorForTokenClassification(tokenizer=tokenizer)
base_model = AutoModelForTokenClassification.from_pretrained(str(BASELINE_DIR))
conll_id2label = {int(k): v for k, v in base_model.config.id2label.items()}

def boundary_zeroshot():
    from torch.utils.data import DataLoader
    test_ds = load_from_disk(str(TOKENIZED_DIR / DS / 'test'))
    dev = 'cuda' if torch.cuda.is_available() else 'cpu'; base_model.to(dev).eval()
    loader = DataLoader(test_ds, batch_size=32, collate_fn=collator)
    true_b, pred_b = [], []
    with torch.no_grad():
        for batch in loader:
            labels = batch.pop('labels')
            logits = base_model(**{k: v.to(dev) for k, v in batch.items()}).logits
            preds = torch.argmax(logits, -1).cpu()
            for pr, lr in zip(preds, labels):
                st, sp = [], []
                for p, l in zip(pr.tolist(), lr.tolist()):
                    if l == -100: continue
                    st.append(collapse(id2label[l])); sp.append(collapse(conll_id2label[p]))
                true_b.append(st); pred_b.append(sp)
    return {'precision': float(precision_score(true_b, pred_b)),
            'recall': float(recall_score(true_b, pred_b)),
            'f1': float(f1_score(true_b, pred_b))}

zs = boundary_zeroshot()
json.dump({'zero_shot_cross_domain_boundary_f1': {DS: zs}},
          open(JN_RESULTS / 'baseline_jnlpba.json', 'w'), indent=2)
print('JNLPBA zero-shot boundary F1:', round(zs['f1'], 3))

JNLPBA zero-shot boundary F1: 0.269


## Step 6 — Few-shot fine-tuning grid (Arm 1, same recipe as Notebook 05)

In [7]:
import gc, time
from transformers import (AutoConfig, AutoTokenizer as _T, EarlyStoppingCallback,
                          Trainer, TrainingArguments, set_seed)

LR, EPOCHS, TRAIN_BS, EVAL_BS, WD, PATIENCE = 2e-5, 8, 8, 32, 0.01, 2
baseline_encoder = AutoModelForTokenClassification.from_pretrained(str(BASELINE_DIR)).base_model.state_dict()

def build_metrics():
    def cm(eval_pred):
        preds, labels = eval_pred
        tt, pp = decode(preds, labels, id2label)
        tb = [[collapse(t) for t in s] for s in tt]; pb = [[collapse(t) for t in s] for s in pp]
        return {'typed_f1': f1_score(tt, pp), 'boundary_f1': f1_score(tb, pb)}
    return cm

def make_model():
    cfg = AutoConfig.from_pretrained('bert-base-cased', num_labels=len(id2label),
                                     id2label=id2label, label2id=label2id)
    m = AutoModelForTokenClassification.from_pretrained('bert-base-cased', config=cfg,
                                                        ignore_mismatched_sizes=True)
    m.base_model.load_state_dict(baseline_encoder, strict=True)
    return m

RESULTS_CSV = JN_RESULTS / 'fewshot_jnlpba.csv'
# matched 800-sentence eval subset -- IDENTICAL selection to Notebook 14 (seed 42)
EVAL_LIMIT, EVAL_SAMPLE_SEED = 800, 42
_ntest = len(load_jsonl(PROCESSED / DS / f'{DS}_test.jsonl'))
sub_idx = (sorted(random.Random(EVAL_SAMPLE_SEED).sample(range(_ntest), EVAL_LIMIT))
           if _ntest > EVAL_LIMIT else list(range(_ntest)))
done = set(); all_results = []
if RESULTS_CSV.exists():
    prev = pd.read_csv(RESULTS_CSV)
    if 'n_eval' in prev.columns:                       # already matched-eval format -> resume
        done = set(zip(prev.dataset, prev.budget, prev.seed)); all_results = prev.to_dict('records')
        print('resuming from', len(done), 'runs')
    else:
        print('old full-test-only results found -> recomputing with the matched 800-subset eval')

for b in BUDGETS:
    for s in SEEDS:
        if (DS, b, s) in done: continue
        print('=' * 60, f'\n{DS} budget={b} seed={s}')
        set_seed(s)
        train_ds = load_from_disk(str(TOKENIZED_DIR / 'fewshot' / DS / f'budget{b}_seed{s}'))
        val_ds = load_from_disk(str(TOKENIZED_DIR / DS / 'validation'))
        test_ds = load_from_disk(str(TOKENIZED_DIR / DS / 'test'))
        model = make_model()
        args = TrainingArguments(output_dir=f'/content/jn_{b}_{s}', eval_strategy='epoch',
                    save_strategy='epoch', logging_strategy='epoch', learning_rate=LR,
                    per_device_train_batch_size=TRAIN_BS, per_device_eval_batch_size=EVAL_BS,
                    num_train_epochs=EPOCHS, weight_decay=WD, load_best_model_at_end=True,
                    metric_for_best_model='typed_f1', greater_is_better=True, save_total_limit=1,
                    report_to='none', seed=s)
        trainer = Trainer(model=model, args=args, train_dataset=train_ds, eval_dataset=val_ds,
                    tokenizer=tokenizer, data_collator=collator, compute_metrics=build_metrics(),
                    callbacks=[EarlyStoppingCallback(early_stopping_patience=PATIENCE)])
        t0 = time.time(); trainer.train(); secs = time.time() - t0
        preds, labels, _ = trainer.predict(test_ds)
        tt, pp = decode(preds, labels, id2label)
        tb = [[collapse(t) for t in x] for x in tt]; pb = [[collapse(t) for t in x] for x in pp]
        # subset matched to Notebook 14 (primary); full test kept as reference columns
        tt_s = [tt[i] for i in sub_idx]; pp_s = [pp[i] for i in sub_idx]
        tb_s = [tb[i] for i in sub_idx]; pb_s = [pb[i] for i in sub_idx]
        row = {'method': 'fewshot', 'dataset': DS, 'budget': b, 'seed': s,
               'num_train_sentences': len(train_ds), 'train_seconds': round(secs, 1),
               'n_eval': len(sub_idx),
               'test_typed_f1': float(f1_score(tt_s, pp_s)), 'test_boundary_f1': float(f1_score(tb_s, pb_s)),
               'per_type': json.dumps(per_type_from(tt_s, pp_s)),
               'test_typed_f1_full': float(f1_score(tt, pp)), 'test_boundary_f1_full': float(f1_score(tb, pb)),
               'per_type_full': json.dumps(per_type_from(tt, pp))}
        all_results.append(row)
        pd.DataFrame(all_results).to_csv(RESULTS_CSV, index=False)
        print(f"  [800-subset] typed F1={row['test_typed_f1']:.3f}  |  [full test] typed F1={row['test_typed_f1_full']:.3f}")
        del trainer, model; gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()

df = pd.read_csv(RESULTS_CSV)
display(df.groupby('budget')[['test_typed_f1', 'test_typed_f1_full', 'test_boundary_f1']].mean().round(3))
print('saved ->', RESULTS_CSV)

old full-test-only results found -> recomputing with the matched 800-subset eval
jnlpba budget=50 seed=13


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/436M [00:00<?, ?B/s]

Some weights of BertForTokenClassification were not initialized from the model checkpoint at bert-base-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


...

Some weights of BertForTokenClassification were not initialized from the model checkpoint at bert-base-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss,Typed F1,Boundary F1
1,0.911500,0.540344,0.295008,0.456907
2,0.458000,0.446262,0.299764,0.520931
3,0.322900,0.384389,0.433239,0.620648
4,0.226500,0.400783,0.465360,0.599957
5,0.179300,0.368472,0.508673,0.639209
6,0.144700,0.364439,0.526958,0.645584
7,0.117000,0.368533,0.548254,0.659597
8,0.100000,0.368732,0.542369,0.656417


  [800-subset] typed F1=0.512  |  [full test] typed F1=0.512


,test_typed_f1,test_typed_f1_full,test_boundary_f1
budget,,,
50,0.312,0.324,0.410
100,0.387,0.397,0.519
200,0.510,0.515,0.611


saved -> /content/drive/MyDrive/AAI590/data/processed/results/jnlpba/fewshot_jnlpba.csv
